In [1]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv("../DAY 1/.env")

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Gemini connected!")

Gemini connected!


In [2]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    title: str
    date: str
    time: str
    attendees: list[str]

In [3]:
user_input = "Schedule a project meeting with Rahul and Priya tomorrow at 3 PM."

prompt = f"""
Extract the calendar event details from the user's request.

Return:
- title
- date
- time
- attendees

User request:
{user_input}
"""

In [5]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json"
    }
)

print(response.text)

{
  "title": "Project Meeting",
  "date": "tomorrow",
  "time": "3:00 PM",
  "attendees": [
    "Rahul",
    "Priya"
  ]
}


In [6]:
import json

data = json.loads(response.text)

event = CalendarEvent.model_validate(data)

print(event)

title='Project Meeting' date='tomorrow' time='3:00 PM' attendees=['Rahul', 'Priya']


In [7]:
def create_calendar_event(title: str, date: str, time: str, attendees: list[str]) -> str:
    return f"Calendar event created: {title} on {date} at {time} with {', '.join(attendees)}"

In [8]:
tool_result = create_calendar_event(
    title=event.title,
    date=event.date,
    time=event.time,
    attendees=event.attendees
)

print(tool_result)

Calendar event created: Project Meeting on tomorrow at 3:00 PM with Rahul, Priya


In [9]:
tools = [create_calendar_event]

In [10]:
user_input_2 = "Set up a team meeting with Anu and Ravi on Friday at 10 AM."

prompt_2 = f"""
Extract the calendar event details from the user's request.

Return:
- title
- date
- time
- attendees

User request:
{user_input_2}
"""

In [11]:
response_2 = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_2,
    config={
        "response_mime_type": "application/json"
    }
)

print(response_2.text)

{
  "title": "Team Meeting",
  "date": "Friday",
  "time": "10:00 AM",
  "attendees": [
    "Anu",
    "Ravi"
  ]
}


In [12]:
data_2 = json.loads(response_2.text)

event_2 = CalendarEvent.model_validate(data_2)

print(event_2)

title='Team Meeting' date='Friday' time='10:00 AM' attendees=['Anu', 'Ravi']


In [13]:
tool_result_2 = create_calendar_event(
    title=event_2.title,
    date=event_2.date,
    time=event_2.time,
    attendees=event_2.attendees
)

print(tool_result_2)

Calendar event created: Team Meeting on Friday at 10:00 AM with Anu, Ravi


In [14]:
response_3 = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Create a calendar event for a project meeting with Anu and Ravi on Friday at 10 AM.",
    config={
        "tools": [create_calendar_event]
    }
)

print(response_3.candidates[0].content.parts)

[Part(
  text='The calendar event for the **Project Meeting** with **Anu** and **Ravi** has been successfully created for **Friday at 10:00 AM**.',
  thought_signature=b'\x12\xa1\x01\n\x9e\x01\x01\x11M2\x0f:j\x0b\x04>\xa9\xda\x0b\xf50~D\x11H1\xcdl\x8d5\x13Dw\xa3\xd2\xc918H\x85~\xc9\xa5\xf7b\xa1j"\xa1\xc29\xdaUE\xf4\xe3AfE\xbd\x84\xfa\xb6o\x92Q\x94\xac\x99$\x7f\x92\xf3.\xe0gt+\x06M\x86<\x01\xa8\xd1}b\x83Er\x9a\'\x8a\x9b\x98:\x89...'
)]


In [15]:
for part in response_3.candidates[0].content.parts:
    print("TEXT:", part.text)
    print("FUNCTION CALL:", part.function_call)

TEXT: The calendar event for the **Project Meeting** with **Anu** and **Ravi** has been successfully created for **Friday at 10:00 AM**.
FUNCTION CALL: None


In [16]:
result = create_calendar_event(
    title=event_2.title,
    date=event_2.date,
    time=event_2.time,
    attendees=event_2.attendees
)

print(result)

Calendar event created: Team Meeting on Friday at 10:00 AM with Anu, Ravi


In [17]:
user_request = input("What would you like to schedule? ")

print("You asked:", user_request)

What would you like to schedule?  Meeting with Karthik tomorrow at 4 PM about the AI project


You asked: Meeting with Karthik tomorrow at 4 PM about the AI project


In [18]:
prompt_3 = f"""
Extract the calendar event details from the user's request.

Return only these fields:
- title
- date
- time
- attendees

User request:
{user_request}
"""

response_4 = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_3,
    config={
        "response_mime_type": "application/json"
    }
)

print(response_4.text)

{
  "title": "Meeting about the AI project",
  "date": "tomorrow",
  "time": "4:00 PM",
  "attendees": [
    "Karthik"
  ]
}


In [19]:
data_3 = json.loads(response_4.text)

event_3 = CalendarEvent.model_validate(data_3)

print(event_3)

title='Meeting about the AI project' date='tomorrow' time='4:00 PM' attendees=['Karthik']


In [20]:
result_3 = create_calendar_event(
    title=event_3.title,
    date=event_3.date,
    time=event_3.time,
    attendees=event_3.attendees
)

print(result_3)

Calendar event created: Meeting about the AI project on tomorrow at 4:00 PM with Karthik


while True:
    user_request = input("\nWhat would you like to schedule? ")

    if user_request.lower() == "exit":
        print("Goodbye!")
        break

    prompt = f"""
Extract the calendar event details from the user's request.

Return only:
- title
- date
- time
- attendees

User request:
{user_request}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    data = json.loads(response.text)

    try:
        event = CalendarEvent.model_validate(data)

        result = create_calendar_event(
            title=event.title,
            date=event.date,
            time=event.time,
            attendees=event.attendees
        )

        print(result)

    except Exception as e:
        print("Could not create the event:", e)

## Conclusion

In this project, I built a Smart Assistant CLI that converts natural-language calendar requests into structured event data.

Gemini extracts the event details as JSON, and Pydantic validates the data against the CalendarEvent schema.

After validation, the Python calendar function is executed using the validated event data.

This project combines the structured output concepts from Day 4 with the tool/function execution concepts from Day 5.

### Key Learning

- Natural language can be converted into structured JSON.
- Pydantic validates the extracted data.
- Validated data can safely be passed to Python functions.
- Tool functions allow an LLM application to perform actions.
- A complete LLM application can combine extraction, validation, and tool execution.

In [ ]:
%%writefile README.md
# Day 07 - Smart Assistant CLI

## What I learned
- How to combine Gemini with structured JSON output.
- How Pydantic validates structured LLM output.
- How validated data can be passed to a Python function.
- How LLM applications can combine extraction, validation, and tool execution.

## What I built
A Smart Assistant CLI that accepts natural-language calendar requests.

For example:

"Meeting with Priya tomorrow at 2 PM about the project"

The assistant extracts the event details, converts them into JSON, validates them with Pydantic, and passes the validated data to a calendar function.

## Pipeline

User Request
→ Gemini
→ Structured JSON
→ Pydantic Validation
→ Calendar Function
→ Event Created

## Key Learning
The project combines structured output from Day 4 with the tool/function execution concepts from Day 5.

The calendar function is simulated in Python and does not connect to a real calendar service.